<a href="https://colab.research.google.com/github/shentan-shiina/Colab_Deployment_Log/blob/main/CLOVER_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import auth
auth.authenticate_user()


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh

downloading uv 0.10.0 x86_64-unknown-linux-gnu
no checksums to verify
installing to /usr/local/bin
  uv
  uvx
everything's installed!



##Install CLOVER
---



In [ ]:
%cd /content

/content


In [ ]:
!git clone --recurse-submodules https://github.com/OpenDriveLab/CLOVER.git

Cloning into 'CLOVER'...
remote: Enumerating objects: 185, done.
remote: Counting objects: 100% (185/185), done.
remote: Compressing objects: 100% (165/165), done.
remote: Total 185 (delta 73), reused 43 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (185/185), 13.82 MiB | 6.37 MiB/s, done.
Resolving deltas: 100% (73/73), done.


## Create py3.8 env first

In [ ]:
!uv venv --python 3.8 /content/CLOVER/.venv
%env PYTHONPATH=$PYTHONPATH:/content/CLOVER/


Using CPython 3.8.20
Creating virtual environment at: CLOVER/.venv
Activate with: source CLOVER/.venv/bin/activate
env: PYTHONPATH=$PYTHONPATH:/content/CLOVER/


In [ ]:
%cd /content/CLOVER
!uv pip install --python /content/CLOVER/.venv/bin/python torch==1.13.1+cu117 torchvision==0.14.1+cu117 torchaudio==0.13.1 --extra-index-url https://download.pytorch.org/whl/cu117

/content/CLOVER
Resolved 11 packages in 3.77s
Prepared 11 packages in 17.24s
Installed 11 packages in 183ms
 + certifi==2022.12.7
 + charset-normalizer==2.1.1
 + idna==3.4
 + numpy==1.24.4
 + pillow==10.4.0
 + requests==2.28.1
 + torch==1.13.1+cu117
 + torchaudio==0.13.1+cu117
 + torchvision==0.14.1+cu117
 + typing-extensions==4.12.2
 + urllib3==1.26.13


In [ ]:
%cd /content/CLOVER/
# !uv pip install --python /content/CLOVER/.venv/bin/python git+https://github.com/hassony2/torch_videovision
# Remove above line in CLOVER's requirements.txt
!uv pip install --python /content/CLOVER/.venv/bin/python -e .

/content/CLOVER
Resolved 107 packages in 897ms
Prepared 1 package in 244ms
Uninstalled 4 packages in 68ms
Installed 4 packages in 29ms
 ~ clover==0.0.1 (from file:///content/CLOVER)
 - huggingface-hub==0.36.1
 + huggingface-hub==0.17.3
 - tokenizers==0.20.3
 + tokenizers==0.14.1
 - transformers==4.46.3
 + transformers==4.34.0




---



In [ ]:
# @title Magic Command for venv
import sys
import os
import subprocess
import time
from IPython.core.magic import register_cell_magic, register_line_cell_magic

# --- CONFIGURATION ---
# Point this to where your UV environment lives
VENV_ROOT = "/content/CLOVER/.venv"
VENV_BIN = os.path.join(VENV_ROOT, "bin")
VENV_ACTIVATE = os.path.join(VENV_BIN, "activate")
# ---------------------

def run_live_output(command, cwd=None):
    """
    Runs a command using subprocess, streaming output, with context awareness.
    """
    env = os.environ.copy()

    # Add the current directory to PYTHONPATH so imports work dynamically
    if cwd:
        current_pythonpath = env.get("PYTHONPATH", "")
        env["PYTHONPATH"] = f"{cwd}:{current_pythonpath}"

    process = subprocess.Popen(
        command,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True,
        env=env,
        cwd=cwd
    )

    while True:
        output_line = process.stdout.readline()
        if output_line == '' and process.poll() is not None:
            break
        if output_line:
            sys.stdout.write(output_line)
            sys.stdout.flush()

    return process.returncode

@register_cell_magic
def py_env(line, cell):
    """
    Executes Python code dynamically in the CURRENT directory.
    """
    # 1. Get the current dynamic path (wherever you are now)
    current_cwd = os.getcwd()

    # 2. Create the temp file IN THE CURRENT DIRECTORY
    # This fixes issues where code relies on __file__ or relative paths
    script_filename = ".colab_temp_script.py"
    script_path = os.path.join(current_cwd, script_filename)

    with open(script_path, "w") as f:
        f.write(cell)

    try:
        python_executable = os.path.join(VENV_BIN, "python")
        print(f"🐍 Executing in: {current_cwd}")

        # Run the script
        cmd = f'"{python_executable}" "{script_filename}"'
        run_live_output(cmd, cwd=current_cwd)

    finally:
        # 3. Clean up immediately
        if os.path.exists(script_path):
            os.remove(script_path)

@register_line_cell_magic
def bash_env(line, cell=None):
    """
    Executes Bash code dynamically in the CURRENT directory.
    """
    current_cwd = os.getcwd()

    if cell is None:
        # Line Magic
        full_cmd = f'/bin/bash -c "source {VENV_ACTIVATE} && {line}"'
        print(f"💻 Executing line in: {current_cwd}")
        run_live_output(full_cmd, cwd=current_cwd)
        return

    # Cell Magic
    script_filename = ".colab_temp_script.sh"
    script_path = os.path.join(current_cwd, script_filename)

    with open(script_path, "w") as f:
        f.write("set -e\n") # Stop on error
        f.write(cell)

    try:
        print(f"💻 Executing block in: {current_cwd}")
        full_cmd = f'/bin/bash -c "source {VENV_ACTIVATE} && /bin/bash {script_filename}"'
        run_live_output(full_cmd, cwd=current_cwd)
    finally:
        if os.path.exists(script_path):
            os.remove(script_path)

print(f"✅ Magic Command for venv Loaded, Use '%%py_env or %%bash_env' to activate.")

✅ Magic Command for venv Loaded, Use '%%py_env or %%bash_env' to activate.


## Install CALVIN


---



In [ ]:
%cd /content/CLOVER

/content/CLOVER


In [ ]:
!git clone --recurse-submodules https://github.com/mees/calvin.git

fatal: destination path 'calvin' already exists and is not an empty directory.


In [ ]:
%env CALVIN_ROOT=/content/CLOVER/calvin

env: CALVIN_ROOT=/content/CLOVER/calvin


**Get rid of pytorch1.13, torchvision, pyhash first**

In [ ]:
%cd /content/CLOVER/calvin
# Manually install pyhash from git (the original pip pyhash 0.9.3 has severe issues for installation)
!uv pip install --python /content/CLOVER/.venv/bin/python wheel cmake==3.18.4
!uv pip install --python /content/CLOVER/.venv/bin/python setuptools==57.5.0
!uv pip install --python /content/CLOVER/.venv/bin/python git+https://github.com/flier/pyfasthash
!uv pip install -e /content/CLOVER/calvin/calvin_env/tacto --python /content/CLOVER/.venv/bin/python
!uv pip install  -e /content/CLOVER/calvin/calvin_env --python /content/CLOVER/.venv/bin/python
!uv pip install  -e /content/CLOVER/calvin/calvin_models --python /content/CLOVER/.venv/bin/python

/content/CLOVER/calvin
Using Python 3.8.20 environment at: /content/CLOVER/.venv
Audited 2 packages in 2ms
Using Python 3.8.20 environment at: /content/CLOVER/.venv
Audited 1 package in 2ms
Using Python 3.8.20 environment at: /content/CLOVER/.venv
Resolved 1 package in 582ms
Audited 1 package in 0.13ms
Using Python 3.8.20 environment at: /content/CLOVER/.venv
Resolved 32 packages in 600ms
Prepared 1 package in 240ms
Uninstalled 1 package in 0.44ms
Installed 1 package in 0.81ms
 ~ tacto==0.0.3 (from file:///content/CLOVER/calvin/calvin_env/tacto)
Using Python 3.8.20 environment at: /content/CLOVER/.venv
Resolved 41 packages in 598ms
Prepared 1 package in 246ms
Uninstalled 1 package in 0.39ms
Installed 1 package in 0.97ms
 ~ calvin-env==0.0.1 (from file:///content/CLOVER/calvin/calvin_env)
Using Python 3.8.20 environment at: /content/CLOVER/.venv
Resolved 100 packages in 643ms
Prepared 1 package in 248ms
Uninstalled 7 packages in 41ms
Installed 19 packages in 46ms
 - antlr4-python3-runti

In [ ]:
# @title Install some missing dependencies before runing feedbackpolicy
!uv pip install --python /content/CLOVER/.venv/bin/python braceexpand
!uv pip install --python /content/CLOVER/.venv/bin/python webdataset
!uv pip install --python /content/CLOVER/.venv/bin/python timm==0.6.11

Audited 1 package in 2ms
Audited 1 package in 2ms
Resolved 36 packages in 307ms
Prepared 1 package in 36ms
Installed 1 package in 5ms
 + timm==0.6.11


## Install the eai-vc repo for VC-1

In [ ]:
%cd /content/CLOVER
!git clone --recurse-submodules https://github.com/facebookresearch/eai-vc.git
!cp -r /content/CLOVER/eai-vc/vc_models/ /content/CLOVER/


/content/CLOVER
fatal: destination path 'eai-vc' already exists and is not an empty directory.


In [ ]:
# Don't know why, but multicoretsne suddenly fails to build
%cd /content/CLOVER/calvin
!uv pip install --python /content/CLOVER/.venv/bin/python MulticoreTSNE

/content/CLOVER/calvin
Using Python 3.8.20 environment at: /content/CLOVER/.venv
Resolved 4 packages in 7ms
  × Failed to build `multicoretsne==0.1`
  ├─▶ The build backend returned an error
  ╰─▶ Call to `setuptools.build_meta:__legacy__.build_wheel` failed (exit
      status: 1)

      [stdout]
      running bdist_wheel
      running build
      running build_py
      copying MulticoreTSNE/__init__.py ->
      build/lib.linux-x86_64-cpython-38/MulticoreTSNE
      copying MulticoreTSNE/tests/__init__.py ->
      build/lib.linux-x86_64-cpython-38/MulticoreTSNE/tests
      copying MulticoreTSNE/tests/test_base.py ->
      build/lib.linux-x86_64-cpython-38/MulticoreTSNE/tests
      running egg_info
      writing MulticoreTSNE.egg-info/PKG-INFO
      writing dependency_links to MulticoreTSNE.egg-info/dependency_links.txt
      writing requirements to MulticoreTSNE.egg-info/requires.txt
      writing top-level names to MulticoreTSNE.egg-info/top_level.txt
      reading manifest file 'Multi

In [ ]:
!uv pip list --python /content/CLOVER/.venv/bin/python | grep setup

Using Python 3.8.20 environment at: /content/CLOVER/.venv
setuptools               57.5.0


In [ ]:
%cd /content/CLOVER/calvin/dataset
!sh download_data.sh debug

/content/CLOVER/calvin/dataset
--2026-02-06 06:11:04--  http://calvin.cs.uni-freiburg.de/dataset/calvin_debug_dataset.zip
Resolving calvin.cs.uni-freiburg.de (calvin.cs.uni-freiburg.de)... 132.230.105.132
Connecting to calvin.cs.uni-freiburg.de (calvin.cs.uni-freiburg.de)|132.230.105.132|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1299150917 (1.2G) [application/zip]
Saving to: ‘calvin_debug_dataset.zip’

calvin_debug_datase 100%[===================>]   1.21G  13.2MB/s    in 1m 48s  

2026-02-06 06:12:53 (11.5 MB/s) - ‘calvin_debug_dataset.zip’ saved [1299150917/1299150917]

Archive:  calvin_debug_dataset.zip
   creating: calvin_debug_dataset/
   creating: calvin_debug_dataset/training/
  inflating: calvin_debug_dataset/training/episode_0359639.npz  
  inflating: calvin_debug_dataset/training/episode_0359404.npz  
  inflating: calvin_debug_dataset/training/episode_0358508.npz  
  inflating: calvin_debug_dataset/training/episode_0358829.npz  
  inflating: cal

In [ ]:
!pip list

Package                               Version             Editable project location
------------------------------------- ------------------- ---------------------------------------
absl-py                               1.4.0
accelerate                            0.23.0
aiofiles                              24.1.0
aiohappyeyeballs                      2.6.1
aiohttp                               3.11.15
aiosignal                             1.4.0
alabaster                             1.0.0
albucore                              0.0.24
albumentations                        2.0.8
ale-py                                0.11.2
altair                                5.5.0
annotated-types                       0.7.0
antlr4-python3-runtime                4.8
anyio                                 4.9.0
appdirs                               1.4.4
argon2-cffi                           25.1.0
argon2-cffi-bindings                  21.2.0
array_record                          0.7.2
arviz               

In [ ]:
%cd /content/CLOVER/

/content/CLOVER


In [ ]:
import sys
import os

# Add the project root to the path
sys.path.append('/content/CLOVER/')

In [ ]:
%env PYTHONPATH=$PYTHONPATH:/content/CLOVER/

env: PYTHONPATH=$PYTHONPATH:/content/CLOVER/


In [ ]:
!nvidia-smi

Fri Feb  6 09:00:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   51C    P8             17W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%%bash_env
# Before this, edit accelerate_cfg to distributed_type: NO
accelerate launch --config_file visual_planner/accelerate_cfg.yaml visual_planner/train.py \
    --learning_rate 1e-4 \
    --train_num_steps 300000 \
    --save_and_sample_every 10000 \
    --train_batch_size 32 \
    --sample_per_seq 8 \
    --sampling_step 5 \
    --with_text_conditioning \
    --diffusion_steps 100 \
    --sample_steps 10 \
    --with_depth \
    --flow_reg \
    --results_folder /content/clover_results

💻 Executing block in: /content/CLOVER
/content/CLOVER/.venv/lib/python3.8/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/content/CLOVER/.venv/lib/python3.8/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/content/CLOVER/.venv/lib/python3.8/site-packages/diffusers/utils/outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/content/CLOVER/.venv/lib/python3.8/site-packages/rotary_embedding_torch/rotary_embedding_torch.py:35: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecate



---



In [ ]:
!touch /content/CLOVER/visual_planner/__init__.py

## Training FeedbackPolicy

In [ ]:
%cd /content/CLOVER/

/content/CLOVER


In [ ]:
# @title
%%bash_env
calvin_dataset_path='/content/CLOVER/calvin/dataset/calvin_debug_dataset'

subfix=`date "+%Y%m%d-%H%M"`
log_file="logs/training_"${subfix}".log"

torchrun --nnodes=1 --nproc_per_node=1 FeedbackPolicy/train/train_calvin.py \
    --vision_encoder vc1-base \
    --num_epochs 10 \
    --gradient_accumulation_steps 1 \
    --batch_size_calvin 16 \
    --run_name feedback_policy_calvin_abc \
    --calvin_dataset ${calvin_dataset_path} \
    --workers 4 \
    --learning_rate 1e-4 \
    --window_size 5 \
    2>&1 | tee ${log_file}

💻 Executing block in: /content/CLOVER
tee: logs/training_20260206-1010.log: No such file or directory
/content/CLOVER/.venv/lib/python3.8/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/content/CLOVER/.venv/lib/python3.8/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/content/CLOVER/.venv/lib/python3.8/site-packages/diffusers/utils/outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/content/CLOVER/.venv/lib/python3.8/site-packages/rotary_embedding_torch/rotary_embedding_torch.py:35

KeyboardInterrupt: 

In [ ]:
!tar -czvf /content/CLOVER/ -C /content/ CLOVER

CLOVER/
CLOVER/CLOVER.egg-info/
tar (child): CLOVER/CLOVER.egg-info/not-zip-safe
/content/CLOVER/: Cannot openCLOVER/CLOVER.egg-info/SOURCES.txt
: Is a directory
CLOVER/CLOVER.egg-info/PKG-INFO
tar (child): Error is not recoverable: exiting now
CLOVER/CLOVER.egg-info/dependency_links.txt
CLOVER/CLOVER.egg-info/requires.txt
CLOVER/CLOVER.egg-info/top_level.txt
CLOVER/README.md
CLOVER/.venv/
CLOVER/.venv/bin/
CLOVER/.venv/bin/f2py3
CLOVER/.venv/bin/deactivate.bat
CLOVER/.venv/bin/python
CLOVER/.venv/bin/tifffile
CLOVER/.venv/bin/activate.nu
CLOVER/.venv/bin/pyrsa-decrypt
CLOVER/.venv/bin/tqdm
CLOVER/.venv/bin/trimesh
CLOVER/.venv/bin/normalizer
CLOVER/.venv/bin/accelerate
CLOVER/.venv/bin/fonttools
CLOVER/.venv/bin/lsm2bin
CLOVER/.venv/bin/activate_this.py
CLOVER/.venv/bin/torchrun
CLOVER/.venv/bin/cpack
CLOVER/.venv/bin/pyrsa-sign
CLOVER/.venv/bin/pydoc.bat
CLOVER/.venv/bin/diffusers-cli
CLOVER/.venv/bin/pyftsubset
CLOVER/.venv/bin/accelerate-launch
CLOVER/.venv/bin/huggingface-cli
CLOV

In [ ]:
!tar -czvf /content/CLOVER_v2.tar.gz -C /content/ CLOVER

Streaming output truncated to the last 5000 lines.
CLOVER/calvin/dataset/calvin_debug_dataset/validation/episode_0553674.npz
CLOVER/calvin/dataset/calvin_debug_dataset/validation/episode_0555047.npz
CLOVER/calvin/dataset/calvin_debug_dataset/validation/episode_0554580.npz
CLOVER/calvin/dataset/calvin_debug_dataset/validation/episode_0553763.npz
CLOVER/calvin/dataset/calvin_debug_dataset/validation/episode_0554257.npz
CLOVER/calvin/dataset/calvin_debug_dataset/validation/episode_0554081.npz
CLOVER/calvin/dataset/calvin_debug_dataset/validation/episode_0554515.npz
CLOVER/calvin/dataset/calvin_debug_dataset/validation/episode_0555156.npz
CLOVER/calvin/dataset/calvin_debug_dataset/validation/episode_0554214.npz
CLOVER/calvin/dataset/calvin_debug_dataset/validation/episode_0554378.npz
CLOVER/calvin/dataset/calvin_debug_dataset/validation/episode_0554556.npz
CLOVER/calvin/dataset/calvin_debug_dataset/validation/episode_0555178.npz
CLOVER/calvin/dataset/calvin_debug_dataset/validation/episode

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# !mkdir /content/drive/MyDrive/Colab/1Environments/CLOVER
!cp /content/CLOVER_v2.tar.gz /content/drive/MyDrive/Colab/1Environments/

In [ ]:
!rm /content/drive/MyDrive/Colab/1Environments/CLOVER/CLOVER.tar.gz

rm: cannot remove '/content/drive/MyDrive/Colab/1Environments/CLOVER/CLOVER.tar.gz': No such file or directory


In [ ]:
!du -sh /content/drive/MyDrive/Colab/1Environments/CLOVER_v2.tar.gz

7.0G	/content/drive/MyDrive/Colab/1Environments/CLOVER_v2.tar.gz


In [ ]:
!uv pip install --python /content/CLOVER/.venv/bin/python torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1 --index-url https://download.pytorch.org/whl/cu121

Resolved 27 packages in 2.85s
Prepared 1 package in 66ms
Uninstalled 1 package in 3ms
Installed 1 package in 5ms
 - torchaudio==0.13.1+cu117
 + torchaudio==2.4.1+cu121


In [ ]:
!/content/CLOVER/.venv/bin/python -c "import torch; print(torch.__version__)"

2.4.1+cu121


In [ ]:
!uv pip list --python /content/CLOVER/.venv/bin/python

Package                  Version     Editable project location
------------------------ ----------- ---------------------------------------
absl-py                  2.3.1
accelerate               0.23.0
aiohappyeyeballs         2.4.4
aiohttp                  3.10.11
aiosignal                1.3.1
antlr4-python3-runtime   4.8
appdirs                  1.4.4
async-timeout            5.0.1
attrs                    25.3.0
beartype                 0.19.0
braceexpand              0.1.7
calvin                   0.0.1       /content/CLOVER/calvin/calvin_models
calvin-env               0.0.1       /content/CLOVER/calvin/calvin_env
certifi                  2022.12.7
cffi                     1.17.1
charset-normalizer       2.1.1
click                    8.1.8
cloudpickle              3.1.2
clover                   0.0.1       /content/CLOVER
cmake                    3.18.4
colorlog                 6.10.1
contourpy                1.1.1
cryptography             45.0.7
cycler                   0.12.1